# CPT-LFR NB0 — Dataset Builder (Final Version)
**No GPU.** Builds all corpora needed for four experiments.

### What this produces
| File | Purpose |
|---|---|
| `control_cpt.jsonl` | Raw C4, length filter only — truly minimal filtering |
| `golden_cpt.jsonl` | C4 + domain keyword filter — same source, one variable changed |
| `val_neutral.jsonl` | WikiText-103 test (neutral — never in training) |
| `val_indomain.jsonl` | Held-out golden docs (in-domain val) |
| `val_ood.jsonl` | Held-out control docs (out-of-domain val) |
| `tier_validation.jsonl` | 20 samples per tier for pre-training check |
| `qa_probe.jsonl` | 60 closed-book domain QA questions |

**After running:** save as Kaggle Dataset `cpt-datasets`

In [1]:
# CELL 1 — Install
import subprocess, sys
for p in ["datasets>=2.14.0","huggingface_hub>=0.19.0","transformers>=4.44.0","tqdm","numpy"]:
    subprocess.run([sys.executable,"-m","pip","install","-q",p],check=True)
    print(f"  ✅ {p}")

  ✅ datasets>=2.14.0
  ✅ huggingface_hub>=0.19.0
  ✅ transformers>=4.44.0
  ✅ tqdm
  ✅ numpy


In [2]:
# CELL 2 — Config
import os, json, random
import numpy as np

random.seed(42); np.random.seed(42)

OUT_DIR     = "/kaggle/working"
CONTROL_N   = 10_000   # raw C4 docs
GOLDEN_N    = 10_000   # domain-filtered C4 docs
VAL_N       = 300      # per split (neutral / in-domain / OOD)
MIN_WORDS   = 100
MAX_WORDS   = 600

# Domain definitions for golden filter
# Key design choice: control and golden use the SAME source (C4)
# Only variable: golden requires ≥2 domain keyword hits
DOMAINS = {
    "mathematics"      : ["theorem","proof","lemma","corollary","conjecture",
                          "integral","derivative","matrix","eigenvalue","manifold",
                          "equation","polynomial","vector","topology","calculus"],
    "computer_science" : ["algorithm","complexity","runtime","recursion","compiler",
                          "cache","concurrency","throughput","latency","heuristic",
                          "binary","pointer","queue","stack","traversal"],
    "natural_sciences" : ["molecule","catalyst","reaction","entropy","wavelength",
                          "photon","electron","nucleus","chromosome","protein",
                          "experiment","hypothesis","specimen","membrane","velocity"],
    "engineering"      : ["torque","voltage","impedance","resonance","bandwidth",
                          "calibration","tolerance","transistor","circuit","sensor",
                          "actuator","hydraulic","thermal","oscillation","filter"],
}
DOMAIN_MIN_HITS = 2   # minimum keyword hits to qualify as domain document
os.makedirs(OUT_DIR, exist_ok=True)
print("Config ready.")
print(f"  Control : {CONTROL_N:,} docs (C4, length filter only)")
print(f"  Golden  : {GOLDEN_N:,} docs (C4, same + domain filter)")
print(f"  Val     : {VAL_N:,} per split × 3 splits")

Config ready.
  Control : 10,000 docs (C4, length filter only)
  Golden  : 10,000 docs (C4, same + domain filter)
  Val     : 300 per split × 3 splits


In [3]:
# CELL 3 — Domain classifier + difficulty scorer
# NOTE: difficulty scoring is LEXICAL — no base model PPL
# Using base model PPL would be circular (scoring difficulty with the thing being trained)

def classify_domain(text):
    """Returns (best_domain, hit_count) or ('other', 0) if no domain matches."""
    text_lower = text.lower()
    scores = {d: sum(text_lower.count(kw) for kw in kws)
              for d, kws in DOMAINS.items()}
    best = max(scores, key=scores.get)
    return (best, scores[best]) if scores[best] >= DOMAIN_MIN_HITS else ("other", 0)

def lexical_difficulty(text):
    """
    3-component lexical difficulty score [0,1].
    No base model perplexity — avoids circularity.
    Component A: avg sentence length (complexity proxy)
    Component B: type-token ratio (lexical diversity)
    Component C: domain keyword density
    """
    words  = text.split()
    sents  = [s.strip() for s in text.split(".") if len(s.strip()) > 10]
    n_w    = max(len(words), 1)
    n_s    = max(len(sents), 1)
    # A
    avg_sl = n_w / n_s
    a      = min(avg_sl / 30.0, 1.0)
    # B
    unique = len(set(w.lower().strip(".,;:!?()") for w in words))
    b      = unique / n_w
    # C
    hits   = sum(sum(text.lower().count(kw) for kw in kws)
                 for kws in DOMAINS.values())
    c      = min(hits / max(n_w / 30, 1), 1.0)
    return round(0.40*a + 0.35*b + 0.25*c, 4)

# Quick test
for text, exp in [
    ("The eigenvalue of a matrix satisfies the characteristic polynomial theorem proof.", "math"),
    ("The compiler optimises algorithm runtime through cache-aware binary traversal.", "cs"),
    ("Today I made a sandwich and watched television all afternoon long.", "other"),
]:
    dom, hits = classify_domain(text)
    diff = lexical_difficulty(text)
    print(f"  [{exp:>5}] domain={dom:<20} hits={hits}  diff={diff}")
print("Classifiers ready ✅")

  [ math] domain=mathematics          hits=5  diff=0.7148
  [   cs] domain=computer_science     hits=6  diff=0.72
  [other] domain=other                hits=0  diff=0.4967
Classifiers ready ✅


In [4]:
# CELL 4 — Build CONTROL corpus (truly minimal filtering)
from datasets import load_dataset

print("Building CONTROL corpus (C4, length filter only)...")
control_docs, scanned = [], 0
for row in load_dataset("allenai/c4","en",split="train",streaming=True):
    text = row["text"].strip()
    words = text.split()
    if len(words) < MIN_WORDS or len(words) > MAX_WORDS: continue
    sents = [s for s in text.split(".") if len(s.strip())>10]
    if len(sents) < 3: continue
    control_docs.append({
        "text"      : text,
        "source"    : "c4_control",
        "quality"   : "control",
        "word_count": len(words),
        "difficulty": lexical_difficulty(text),
        "domain"    : "none",  # no domain filter for control
    })
    scanned += 1
    if len(control_docs) >= CONTROL_N + VAL_N: break
    if scanned % 100_000 == 0:
        print(f"  {scanned:,} scanned → {len(control_docs):,} collected")

random.shuffle(control_docs)
val_ood_docs  = control_docs[:VAL_N]       # OOD val from control
control_train = control_docs[VAL_N:VAL_N+CONTROL_N]
print(f"  Control train : {len(control_train):,}")
print(f"  OOD val       : {len(val_ood_docs):,}")

Building CONTROL corpus (C4, length filter only)...


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

  Control train : 10,000
  OOD val       : 300


In [5]:
# CELL 5 — Build GOLDEN corpus (same source, one variable: domain filter)
print("Building GOLDEN corpus (C4 + domain filter)...")
golden_raw, skipped_other, scanned = [], 0, 0
for row in load_dataset("allenai/c4","en",split="train",streaming=True):
    text  = row["text"].strip()
    words = text.split()
    sents = [s for s in text.split(".") if len(s.strip()) > 10]   # ← ADD THIS
    if len(sents) < 3: continue                                    # ← AND THIS
    dom, hits = classify_domain(text)
    if dom == "other":
        skipped_other += 1
        continue
    golden_raw.append({
        "text"      : text,
        "source"    : "c4_golden",
        "quality"   : "gold",
        "domain"    : dom,
        "word_count": len(words),
        "difficulty": lexical_difficulty(text),
    })
    scanned += 1
    if len(golden_raw) >= GOLDEN_N + VAL_N: break
    if scanned % 200_000 == 0:
        print(f"  {scanned:,} scanned → {len(golden_raw):,} golden (skipped {skipped_other:,})")

random.shuffle(golden_raw)
val_indomain_docs = golden_raw[:VAL_N]
golden_train      = golden_raw[VAL_N:VAL_N+GOLDEN_N]
print(f"  Golden train  : {len(golden_train):,}")
print(f"  In-domain val : {len(val_indomain_docs):,}")
from collections import Counter
dom_dist = Counter(d["domain"] for d in golden_train)
print(f"  Domain dist   : {dict(dom_dist)}")

Building GOLDEN corpus (C4 + domain filter)...


Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

  Golden train  : 10,000
  In-domain val : 300
  Domain dist   : {'natural_sciences': 3941, 'computer_science': 1552, 'mathematics': 1574, 'engineering': 2933}


In [6]:
# CELL 6 — Assign LFR phases (percentile-based — CANNOT produce empty phases)
# This is the fix for the original bug that crashed E4
def assign_lfr_phases(docs, label=""):
    scores = [d["difficulty"] for d in docs]
    p33 = float(np.percentile(scores, 33.33))
    p66 = float(np.percentile(scores, 66.67))
    for d in docs:
        s = d["difficulty"]
        d["lfr_phase"] = "easy" if s<=p33 else ("medium" if s<=p66 else "hard")
    ph = [d["lfr_phase"] for d in docs]
    n_e,n_m,n_h = ph.count("easy"),ph.count("medium"),ph.count("hard")
    assert n_e>0 and n_m>0 and n_h>0, "Empty phase — impossible with percentiles"
    print(f"  {label} LFR (p33={p33:.4f}, p66={p66:.4f}): "
          f"easy={n_e} medium={n_m} hard={n_h}")
    return docs, p33, p66

control_train, cp33, cp66 = assign_lfr_phases(control_train, "Control")
golden_train,  gp33, gp66 = assign_lfr_phases(golden_train,  "Golden")

# Save tier validation samples (20 per tier from golden — used to verify ordering before training)
tier_val_docs = []
for phase in ["easy","medium","hard"]:
    phase_docs = [d for d in golden_train if d["lfr_phase"]==phase]
    tier_val_docs += random.sample(phase_docs, min(20, len(phase_docs)))
print(f"  Tier validation: {len(tier_val_docs)} samples saved for pre-training check")

  Control LFR (p33=0.4355, p66=0.5052): easy=3335 medium=3333 hard=3332
  Golden LFR (p33=0.4423, p66=0.5400): easy=3338 medium=3329 hard=3333
  Tier validation: 60 samples saved for pre-training check


In [7]:
# CELL 7 — Load WikiText-103 neutral validation set
from datasets import load_dataset

print("Loading WikiText-103 test split (neutral val)...")
wt = load_dataset("wikitext","wikitext-103-raw-v1",split="test")
val_neutral = []
for row in wt:
    t = row["text"].strip()
    w = t.split()
    if len(w) < MIN_WORDS or t.startswith("=") or t.endswith("="): continue
    val_neutral.append({"text":t,"source":"wikitext103_test"})
    if len(val_neutral) >= VAL_N: break
print(f"  Neutral val: {len(val_neutral):,} paragraphs")

Loading WikiText-103 test split (neutral val)...


README.md: 0.00B [00:00, ?B/s]

wikitext-103-raw-v1/test-00000-of-00001.(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-103-raw-v1/train-00000-of-00002(…):   0%|          | 0.00/157M [00:00<?, ?B/s]

wikitext-103-raw-v1/train-00001-of-00002(…):   0%|          | 0.00/157M [00:00<?, ?B/s]

wikitext-103-raw-v1/validation-00000-of-(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/1801350 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

  Neutral val: 300 paragraphs


In [8]:
# CELL 8 — Build domain QA probe set (60 questions: 30 CS + 30 math)
# These are closed-book questions whose answers appear in domain training text.
# Used to measure knowledge acquisition beyond just perplexity improvement.

QA_PROBE = [
    # ── Computer Science (30 questions) ──────────────────────────────
    {"domain":"cs","q":"What does Big-O notation O(n log n) describe about an algorithm?",
     "a":"The runtime grows proportionally to n times the logarithm of n",
     "distractors":["Runtime is constant","Runtime grows as n squared","Runtime is logarithmic only"]},
    {"domain":"cs","q":"What is the key property of a binary search tree?",
     "a":"Left subtree nodes are smaller and right subtree nodes are larger than the root",
     "distractors":["All nodes have exactly two children","The tree is always balanced","Nodes are inserted in sorted order"]},
    {"domain":"cs","q":"In merge sort, what happens during the merge phase?",
     "a":"Two sorted subarrays are compared element by element and merged into one sorted array",
     "distractors":["Elements are swapped in place","The array is split recursively","A pivot element is selected"]},
    {"domain":"cs","q":"What is cache locality and why does it matter for algorithm performance?",
     "a":"Accessing memory locations close together reduces cache misses and speeds up computation",
     "distractors":["It means storing data in the cloud","It refers to geographic server location","It describes CPU clock speed"]},
    {"domain":"cs","q":"What does amortised O(1) mean for a dynamic array push operation?",
     "a":"Individual operations may be slow occasionally but average cost per operation is constant",
     "distractors":["Every push is always O(1)","The array never needs resizing","Memory is pre-allocated infinitely"]},
    {"domain":"cs","q":"What is the difference between a stack and a queue data structure?",
     "a":"A stack is LIFO (last in first out) while a queue is FIFO (first in first out)",
     "distractors":["Both are LIFO","A stack is faster than a queue","A queue stores more elements"]},
    {"domain":"cs","q":"What is tail recursion and why do compilers optimise it?",
     "a":"When a recursive call is the last operation, compilers can reuse the stack frame instead of adding a new one",
     "distractors":["Recursion that goes backwards","Recursion with no base case","Recursion using iteration"]},
    {"domain":"cs","q":"What does a hash function collision mean?",
     "a":"Two different inputs produce the same hash output",
     "distractors":["The hash function crashes","Two inputs are identical","The hash table is full"]},
    {"domain":"cs","q":"What is the time complexity of inserting into a balanced binary search tree?",
     "a":"O(log n) because the tree height is logarithmic in the number of nodes",
     "distractors":["O(n)","O(1)","O(n²)"]},
    {"domain":"cs","q":"What is dynamic programming and what problem does it solve?",
     "a":"Storing solutions to overlapping subproblems to avoid redundant computation",
     "distractors":["Dynamically allocating memory","Programming with changing requirements","Runtime code generation"]},
    {"domain":"cs","q":"What is the purpose of a compiler's lexer (tokeniser)?",
     "a":"Converting raw source code characters into a sequence of meaningful tokens",
     "distractors":["Compiling tokens to machine code","Optimising loop performance","Managing memory allocation"]},
    {"domain":"cs","q":"What is a deadlock in concurrent programming?",
     "a":"Two or more threads wait indefinitely for resources held by each other",
     "distractors":["A thread running too slowly","Memory being overwritten","A program that terminates early"]},
    {"domain":"cs","q":"What does graph traversal breadth-first search (BFS) guarantee that DFS does not?",
     "a":"BFS finds the shortest path in an unweighted graph",
     "distractors":["BFS is always faster","BFS uses less memory","BFS visits all nodes twice"]},
    {"domain":"cs","q":"What is the role of a garbage collector in memory management?",
     "a":"Automatically freeing memory that is no longer reachable by the program",
     "distractors":["Clearing screen output","Optimising CPU cache","Compressing stored files"]},
    {"domain":"cs","q":"What is the difference between concurrency and parallelism?",
     "a":"Concurrency is managing multiple tasks that can overlap in time; parallelism is executing them simultaneously on multiple cores",
     "distractors":["They are the same thing","Parallelism is sequential","Concurrency requires multiple CPUs"]},
    {"domain":"cs","q":"What is a pointer in C and what can go wrong with it?",
     "a":"A variable storing a memory address; misuse causes dangling pointers, buffer overflows, and segmentation faults",
     "distractors":["A variable storing a string","A type of loop","A compiler directive"]},
    {"domain":"cs","q":"What does throughput measure in a computing system?",
     "a":"The number of tasks or operations completed per unit of time",
     "distractors":["The time to complete one task","The amount of memory used","The number of CPU cores"]},
    {"domain":"cs","q":"What is the halting problem and why is it significant?",
     "a":"It is undecidable — no algorithm can determine whether an arbitrary program will halt or run forever",
     "distractors":["A bug that stops a program","An optimisation technique","A type of infinite loop"]},
    {"domain":"cs","q":"What is the purpose of a lookup table (LUT) in algorithm design?",
     "a":"Pre-computing results for all possible inputs to replace expensive computation with fast lookups",
     "distractors":["Storing user data","Creating database indexes","Sorting large arrays"]},
    {"domain":"cs","q":"What property makes quicksort have O(n²) worst-case time complexity?",
     "a":"When the pivot is always the smallest or largest element, the partition is maximally unbalanced",
     "distractors":["When the array is already sorted in the correct order only","When memory is insufficient","When there are duplicate elements"]},
    {"domain":"cs","q":"What is a heuristic algorithm?",
     "a":"An algorithm that finds a good approximate solution quickly when an exact solution is computationally infeasible",
     "distractors":["An algorithm that always finds the optimal solution","A random algorithm","An algorithm that never terminates"]},
    {"domain":"cs","q":"What is the difference between latency and throughput?",
     "a":"Latency is the delay for a single operation; throughput is the rate of completing many operations",
     "distractors":["They measure the same thing","Latency is always higher","Throughput only applies to networks"]},
    {"domain":"cs","q":"What is a semaphore in operating systems?",
     "a":"A signalling mechanism that controls access to shared resources by multiple processes",
     "distractors":["A type of memory","A file format","A network protocol"]},
    {"domain":"cs","q":"Why does recursion risk a stack overflow?",
     "a":"Each recursive call adds a new frame to the call stack; deep recursion exhausts available stack memory",
     "distractors":["Recursion is always infinite","Recursive functions run slowly","The heap gets corrupted"]},
    {"domain":"cs","q":"What is the key insight behind the divide-and-conquer algorithmic paradigm?",
     "a":"Breaking a problem into smaller subproblems, solving each independently, then combining the results",
     "distractors":["Solving problems using brute force","Finding the optimal pivot","Avoiding recursion entirely"]},
    {"domain":"cs","q":"What is memoisation?",
     "a":"Caching the results of function calls so the same computation is not repeated for the same inputs",
     "distractors":["A type of sorting","Writing code from memory","Optimising network calls"]},
    {"domain":"cs","q":"What is a race condition?",
     "a":"When the outcome of a computation depends on the unpredictable timing of concurrent thread execution",
     "distractors":["A sorting algorithm","A hardware failure","A compiler warning"]},
    {"domain":"cs","q":"What does it mean for an algorithm to be NP-complete?",
     "a":"It is in NP and every NP problem reduces to it; no known polynomial-time solution exists",
     "distractors":["It runs in linear time","It has no solution","It only works on specific hardware"]},
    {"domain":"cs","q":"What is a spanning tree of a graph?",
     "a":"A subgraph that includes all vertices connected with the minimum number of edges and no cycles",
     "distractors":["A tree with unlimited branches","A complete copy of the graph","A path through all edges"]},
    {"domain":"cs","q":"What is the purpose of virtual memory?",
     "a":"Allowing programs to use more memory than physically available by mapping addresses to disk storage",
     "distractors":["Making RAM faster","Sharing files between users","Compressing program binaries"]},

    # ── Mathematics (30 questions) ────────────────────────────────
    {"domain":"math","q":"What does the fundamental theorem of calculus state?",
     "a":"Differentiation and integration are inverse operations; a definite integral can be computed using an antiderivative",
     "distractors":["All functions are differentiable","Integration always increases a function","Every polynomial has a root"]},
    {"domain":"math","q":"What is an eigenvalue of a matrix?",
     "a":"A scalar λ such that Av = λv for some non-zero vector v, where A is the matrix",
     "distractors":["The determinant of the matrix","The trace of the matrix","A diagonal element"]},
    {"domain":"math","q":"What is the difference between a proof by contradiction and a direct proof?",
     "a":"A direct proof derives the conclusion from premises; proof by contradiction assumes the negation and derives a contradiction",
     "distractors":["They are equivalent always","Contradiction proofs are invalid","Direct proofs are always shorter"]},
    {"domain":"math","q":"What does the intermediate value theorem guarantee?",
     "a":"If a continuous function takes values a and b then it takes every value between a and b",
     "distractors":["Every function has a maximum","Differentiable functions are continuous","All polynomials have integer roots"]},
    {"domain":"math","q":"What is a manifold in mathematics?",
     "a":"A topological space that locally resembles Euclidean space near every point",
     "distractors":["A type of matrix","A function from R to R","An infinite series"]},
    {"domain":"math","q":"What does it mean for a series to converge?",
     "a":"The sequence of partial sums approaches a finite limit as the number of terms grows without bound",
     "distractors":["Every term is positive","All terms decrease","The series has finitely many terms"]},
    {"domain":"math","q":"What is the definition of a group in abstract algebra?",
     "a":"A set with an associative binary operation, an identity element, and inverses for all elements",
     "distractors":["A set of numbers","A type of graph","A polynomial ring"]},
    {"domain":"math","q":"What is L'Hôpital's rule used for?",
     "a":"Evaluating limits of indeterminate forms like 0/0 or ∞/∞ by differentiating numerator and denominator",
     "distractors":["Solving differential equations","Finding eigenvalues","Computing integrals numerically"]},
    {"domain":"math","q":"What does the rank of a matrix tell you?",
     "a":"The dimension of the column space — the number of linearly independent columns",
     "distractors":["The size of the matrix","The number of non-zero entries","The trace of the matrix"]},
    {"domain":"math","q":"What is a bijective function?",
     "a":"A function that is both injective (one-to-one) and surjective (onto)",
     "distractors":["A function that is its own inverse","A function defined on all reals","A linear function"]},
    {"domain":"math","q":"What is the gradient of a multivariable function?",
     "a":"The vector of partial derivatives pointing in the direction of steepest increase",
     "distractors":["The second derivative","The integral of the function","A scalar value"]},
    {"domain":"math","q":"What does Bayes' theorem compute?",
     "a":"The posterior probability of a hypothesis given observed evidence, using prior probability and likelihood",
     "distractors":["The expected value of a random variable","The variance of a distribution","A sum of probabilities"]},
    {"domain":"math","q":"What is the determinant of a matrix used for?",
     "a":"Determining invertibility, computing volume scaling, and solving linear systems via Cramer's rule",
     "distractors":["Finding eigenvalues only","Transposing the matrix","Adding rows together"]},
    {"domain":"math","q":"What is the difference between a lemma and a theorem?",
     "a":"A lemma is a preparatory result used to prove a larger theorem; both are rigorously proven statements",
     "distractors":["A lemma is unproven","Theorems are more specific","They apply in different fields only"]},
    {"domain":"math","q":"What is the chain rule in calculus?",
     "a":"The derivative of a composite function f(g(x)) is f'(g(x)) times g'(x)",
     "distractors":["The rule for integrating products","The sum of derivatives","The rule for constant multiples"]},
    {"domain":"math","q":"What is a corollary?",
     "a":"A result that follows directly and easily from a previously proven theorem",
     "distractors":["A disproven conjecture","A type of axiom","An unrelated theorem"]},
    {"domain":"math","q":"What does it mean for two vectors to be orthogonal?",
     "a":"Their dot product is zero — they are perpendicular to each other",
     "distractors":["They have equal length","They point in the same direction","They span the same space"]},
    {"domain":"math","q":"What is a Taylor series?",
     "a":"An infinite series of terms computed from the derivatives of a function at a single point to approximate it",
     "distractors":["A type of Fourier transform","A method of integration","A way to factor polynomials"]},
    {"domain":"math","q":"What is the null space of a matrix?",
     "a":"The set of all vectors x such that Ax equals the zero vector",
     "distractors":["The zero matrix","The set of eigenvalues","The transpose of the matrix"]},
    {"domain":"math","q":"What is a conjecture?",
     "a":"A mathematical statement believed to be true based on evidence but not yet formally proven",
     "distractors":["A proven theorem","A definition","An axiom"]},
    {"domain":"math","q":"What does convergence in probability mean?",
     "a":"A sequence of random variables converges to a value X if the probability of being far from X approaches zero",
     "distractors":["All values are identical","The variance approaches one","The mean approaches infinity"]},
    {"domain":"math","q":"What is an isomorphism between two groups?",
     "a":"A bijective homomorphism that preserves group structure — the groups are algebraically identical",
     "distractors":["A subset relationship","A type of matrix","A diagonal transformation"]},
    {"domain":"math","q":"What is the integral test for series convergence?",
     "a":"A positive decreasing series converges if and only if the corresponding improper integral converges",
     "distractors":["All series with decreasing terms converge","Only geometric series converge","Divergence requires all terms to increase"]},
    {"domain":"math","q":"What is a topological space?",
     "a":"A set with a collection of open subsets satisfying axioms about unions, intersections, and the whole set",
     "distractors":["A set with a distance function","A type of manifold only","A geometric shape"]},
    {"domain":"math","q":"What does the spectral theorem state for symmetric matrices?",
     "a":"Every real symmetric matrix can be diagonalised by an orthogonal matrix with real eigenvalues",
     "distractors":["All matrices are symmetric","Symmetric matrices have no eigenvalues","The determinant is always positive"]},
    {"domain":"math","q":"What is the difference between a rational and irrational number?",
     "a":"A rational number can be expressed as a ratio of integers; an irrational number cannot",
     "distractors":["Irrational numbers are negative","Rational numbers are always integers","Irrational numbers are imaginary"]},
    {"domain":"math","q":"What is induction as a proof technique?",
     "a":"Proving a base case then proving that if the statement holds for n it holds for n+1, establishing it for all n",
     "distractors":["Testing all possible values","Proof by contradiction only","Assuming the conclusion"]},
    {"domain":"math","q":"What is a partial derivative?",
     "a":"The derivative of a multivariable function with respect to one variable, treating all others as constants",
     "distractors":["A derivative of a partial function","An incomplete derivative","The gradient vector"]},
    {"domain":"math","q":"What does the Pythagorean theorem state about right triangles?",
     "a":"The square of the hypotenuse equals the sum of squares of the other two sides",
     "distractors":["All angles sum to 180 degrees","The longest side is always vertical","Area equals half base times height"]},
    {"domain":"math","q":"What is a field in abstract algebra?",
     "a":"A set with two operations (addition and multiplication) where both form abelian groups and multiplication distributes over addition",
     "distractors":["A set of vectors","A type of matrix","A topological space"]},
]

assert len([q for q in QA_PROBE if q["domain"]=="cs"])   == 30
assert len([q for q in QA_PROBE if q["domain"]=="math"]) == 30
print(f"QA probe: {len(QA_PROBE)} questions (30 CS + 30 math) ✅")

QA probe: 60 questions (30 CS + 30 math) ✅


In [9]:
# CELL 9 — Save all files + verification
def save_jsonl(data, fname):
    path = f"{OUT_DIR}/{fname}"
    with open(path,"w") as f:
        for d in data: f.write(json.dumps(d)+"\n")
    print(f"  ✅ {fname}: {len(data):,} records ({os.path.getsize(path)//1024} KB)")

save_jsonl(control_train,    "control_cpt.jsonl")
save_jsonl(golden_train,     "golden_cpt.jsonl")
save_jsonl(val_neutral,      "val_neutral.jsonl")
save_jsonl(val_indomain_docs,"val_indomain.jsonl")
save_jsonl(val_ood_docs,     "val_ood.jsonl")
save_jsonl(tier_val_docs,    "tier_validation.jsonl")
save_jsonl(QA_PROBE,         "qa_probe.jsonl")

# Save LFR thresholds for reference
thresholds = {
    "control": {"p33": cp33, "p66": cp66},
    "golden":  {"p33": gp33, "p66": gp66}
}
with open(f"{OUT_DIR}/lfr_thresholds.json","w") as f:
    json.dump(thresholds, f, indent=2)
print("  ✅ lfr_thresholds.json")

print("\n📊 Summary:")
for fname in ["control_cpt.jsonl","golden_cpt.jsonl"]:
    docs = [json.loads(l) for l in open(f"{OUT_DIR}/{fname}")]
    diffs = [d["difficulty"] for d in docs]
    ph = [d["lfr_phase"] for d in docs]
    print(f"  {fname}: mean_diff={np.mean(diffs):.3f}  "
          f"easy={ph.count('easy')} med={ph.count('medium')} hard={ph.count('hard')}")
print("\n✅ NB0 complete — save output as Kaggle Dataset: 'cpt-datasets'")

  ✅ control_cpt.jsonl: 10,000 records (17372 KB)
  ✅ golden_cpt.jsonl: 10,000 records (69443 KB)
  ✅ val_neutral.jsonl: 300 records (272 KB)
  ✅ val_indomain.jsonl: 300 records (2021 KB)
  ✅ val_ood.jsonl: 300 records (543 KB)
  ✅ tier_validation.jsonl: 60 records (309 KB)
  ✅ qa_probe.jsonl: 60 records (16 KB)
  ✅ lfr_thresholds.json

📊 Summary:
  control_cpt.jsonl: mean_diff=0.475  easy=3335 med=3333 hard=3332
  golden_cpt.jsonl: mean_diff=0.503  easy=3338 med=3329 hard=3333

✅ NB0 complete — save output as Kaggle Dataset: 'cpt-datasets'
